In [ ]:
import sys
import subprocess

def ensure_installed(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

ensure_installed(['transformers', 'datasets', 'torch', 'scikit-learn', 'pandas', 'numpy'])

In [ ]:
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, classification_report

if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
    device_name = 'mps'
else:
    device = torch.device('cpu')
    device_name = 'cpu'

print({'selected_device': device_name})

In [ ]:
dataset = load_dataset('dair-ai/emotion', split='test')
class_names = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

prototype_sentences = {
    'sadness': 'I feel sad and heartbroken.',
    'joy': 'I feel happy and joyful.',
    'love': 'I feel deeply loved and affectionate.',
    'anger': 'I feel angry and frustrated.',
    'fear': 'I feel afraid and anxious.',
    'surprise': 'I feel surprised and amazed.'
}

print({'split': 'test', 'num_rows': len(dataset)})
print(dataset[:3])

In [ ]:
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()

print({'model_name': model_name, 'hidden_size': int(model.config.hidden_size)})

In [ ]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, dim=1) / torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)

def encode_texts(texts, batch_size=64, max_length=128):
    all_embeddings = []
    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.no_grad():
            model_output = model(**encoded)
            sentence_embeddings = mean_pooling(model_output, encoded['attention_mask'])
            sentence_embeddings = torch.nn.functional.normalize(sentence_embeddings, p=2, dim=1)
        all_embeddings.append(sentence_embeddings.cpu())
    return torch.cat(all_embeddings, dim=0)

prototype_texts = [prototype_sentences[name] for name in class_names]
prototype_embeddings = encode_texts(prototype_texts, batch_size=len(prototype_texts), max_length=32)

print({'num_prototypes': len(prototype_texts), 'embedding_shape': list(prototype_embeddings.shape)})
print(dict(zip(class_names, prototype_texts)))

In [ ]:
texts = dataset['text']
true_ids = np.array(dataset['label'])

text_embeddings = encode_texts(texts, batch_size=64, max_length=128)
similarity = text_embeddings @ prototype_embeddings.T
pred_ids = similarity.argmax(dim=1).numpy()
pred_labels = [class_names[i] for i in pred_ids]

results_df = pd.DataFrame({
    'text': texts,
    'true_label': [class_names[i] for i in true_ids],
    'predicted_label': pred_labels,
    'max_similarity': similarity.max(dim=1).values.numpy()
})

print(results_df.head(10).to_dict(orient='records'))

In [ ]:
accuracy = accuracy_score(true_ids, pred_ids)
report = classification_report(true_ids, pred_ids, target_names=class_names, digits=4)

print({
    'model_name': model_name,
    'dataset': 'dair-ai/emotion',
    'split': 'test',
    'num_examples': len(dataset),
    'device': device_name,
    'method': 'nearest_single_prototype_by_cosine_similarity',
    'accuracy': round(float(accuracy), 6)
})
print(report)

In [ ]:
sample_n = 8
print(results_df.head(sample_n).to_string(index=False))